# Spatial Regime Structure of Optical Vortices in a Stribeck-Coupled Medium

N. Joven — April 2026

## Motivation

A vortex in a medium has a velocity field v(r) that varies with distance from the core.
The Stribeck curve gives velocity-dependent coupling K(v): strong coupling (stick) at
low velocity, weak coupling (slip) at high velocity. Composing these:

$$K(r) = K_{\text{stribeck}}(v(r))$$

The vortex core has zero flow velocity, hence maximum coupling (K = K_static).
If K_static > 1, the core is in the overcritical regime of the circle map. The
critical radius r_c, where K(r_c) = 1, separates:

- **r < r_c** (core): K > 1, fold active, reversed energy flow
- **r > r_c** (exterior): K < 1, mode-locked, forward energy flow

This structure predicts a subwavelength reversal boundary that can be compared to
the observed extent of reverse Poynting vector flow near optical vortex cores
(Kotlyar et al., Opt. Lett. 2026; Pryamikov, arXiv 2601.21704).

## References

1. Pryamikov, "Reverse Energy Flows in 2D Photonic Crystals," arXiv:2601.21704 (2026)
2. Kotlyar et al., "2D optical vortices and reverse energy flow," Opt. Lett. 51(4) (2026)
3. Bucher et al., "Superluminal Correlations in Phase Singularities," Nature (2026), arXiv:2509.17675
4. Arnold, *Geometrical Methods in ODE* (1983), ch. 11
5. Stribeck, Z. Verein. Deutsch. Ing. 46 (1902)
6. harmonics/driven_stribeck.py — Stribeck friction implementation
7. harmonics/sync_cost/FRAMEWORK.md — synchronization cost formulation
8. proslambenomenos/kuramoto_einstein_mapping.md — Kuramoto-Einstein dictionary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# Stribeck coupling: K(v)
def stribeck_K(v, K_static=1.8, K_kinetic=0.3, v_threshold=1.0):
    return K_kinetic + (K_static - K_kinetic) * np.exp(-(np.abs(v) / v_threshold)**2)

# Laguerre-Gaussian vortex velocity: v(r, ell)
def vortex_velocity(r, ell=1, wavelength=1.0):
    r_core = wavelength / (2 * np.pi)
    return ell * r / (r**2 + r_core**2)

# Composed regime map: K(r) = K_stribeck(v(r))
def regime_map(r, ell=1, wavelength=1.0, **kw):
    return stribeck_K(vortex_velocity(r, ell, wavelength), **kw)

# Fold measure: mu = arccos(1/K)/pi for K > 1, else 0
def fold_measure(K):
    K = np.asarray(K, dtype=float)
    mu = np.zeros_like(K)
    mask = K > 1.0
    mu[mask] = np.arccos(1.0 / K[mask]) / np.pi
    return mu

# Lyapunov exponent of the standard circle map
def circle_map_lyapunov(Omega, K, n_iter=5000, n_transient=1000):
    theta = 0.5
    for _ in range(n_transient):
        theta = (theta + Omega - (K/(2*np.pi)) * np.sin(2*np.pi*theta)) % 1.0
    log_sum = 0.0
    for _ in range(n_iter):
        deriv = 1.0 - K * np.cos(2*np.pi*theta)
        log_sum += np.log(max(abs(deriv), 1e-15))
        theta = (theta + Omega - (K/(2*np.pi)) * np.sin(2*np.pi*theta)) % 1.0
    return log_sum / n_iter

# Find r_c where K(r) = 1
def find_critical_radii(ell=1, wavelength=1.0, **kw):
    r_core = wavelength / (2*np.pi)
    r_search = np.linspace(1e-6, 20*r_core, 100000)
    K_vals = regime_map(r_search, ell=ell, wavelength=wavelength, **kw)
    crossings = []
    for i in range(len(K_vals)-1):
        if (K_vals[i]-1)*(K_vals[i+1]-1) < 0:
            rc = brentq(lambda r: regime_map(r, ell=ell, wavelength=wavelength, **kw) - 1,
                        r_search[i], r_search[i+1])
            crossings.append(rc)
    return crossings

## 1. Regime Map K(r) for Multiple Topological Charges

The Stribeck parameters used here are illustrative. For a specific photonic crystal,
K_static, K_kinetic, and v_threshold would be extracted from the electromagnetic
coupling structure. The qualitative features — overcritical core, subcritical exterior,
monotonic r_c dependence on ell — are robust to parameter choice provided K_static > 1.

In [ ]:
wavelength = 1.0
r_core = wavelength / (2*np.pi)
K_static, K_kinetic, v_threshold = 1.8, 0.3, 1.0
kw = dict(K_static=K_static, K_kinetic=K_kinetic, v_threshold=v_threshold)

r = np.linspace(0.001, 5*r_core, 2000)
charges = [1, 2, 3, 5, 8]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# (a) Velocity field
ax = axes[0, 0]
for ell in charges:
    ax.plot(r/r_core, vortex_velocity(r, ell, wavelength), label=f'$\\ell = {ell}$')
ax.axhline(v_threshold, color='gray', ls='--', alpha=0.5, label=f'$v_{{thr}}$')
ax.set_xlabel('$r / r_{core}$'); ax.set_ylabel('$v(r)$')
ax.set_title('(a) Azimuthal velocity field'); ax.legend(fontsize=8); ax.set_xlim(0, 5)

# (b) K(r)
ax = axes[0, 1]
for ell in charges:
    ax.plot(r/r_core, regime_map(r, ell, wavelength, **kw), label=f'$\\ell = {ell}$')
ax.axhline(1.0, color='red', ls='-', alpha=0.7, lw=2, label='$K = 1$')
ax.set_xlabel('$r / r_{core}$'); ax.set_ylabel('$K(r)$')
ax.set_title('(b) Coupling regime map'); ax.legend(fontsize=8)
ax.set_xlim(0, 5); ax.set_ylim(0, 2)
ax.fill_between([0, 5], 1, 2, alpha=0.05, color='red')
ax.text(0.15, 1.5, '$K > 1$: fold, reversed flow', fontsize=9, color='red', alpha=0.7)
ax.text(0.15, 0.6, '$K < 1$: mode-locked', fontsize=9, color='blue', alpha=0.7)

# (c) Fold measure
ax = axes[1, 0]
for ell in charges:
    K = regime_map(r, ell, wavelength, **kw)
    ax.plot(r/r_core, fold_measure(K), label=f'$\\ell = {ell}$')
ax.set_xlabel('$r / r_{core}$'); ax.set_ylabel('$\\mu(r)$')
ax.set_title('(c) Fold measure (reversed flow fraction)'); ax.legend(fontsize=8); ax.set_xlim(0, 5)

# (d) Critical radii vs charge
ax = axes[1, 1]
ell_range = np.arange(1, 21)
inner_rc, outer_rc = [], []
for ell in ell_range:
    cx = find_critical_radii(ell, wavelength, **kw)
    inner_rc.append(cx[0]/r_core if len(cx) >= 1 else np.nan)
    outer_rc.append(cx[1]/r_core if len(cx) >= 2 else np.nan)
ax.plot(ell_range, inner_rc, 'o-', ms=4, label='Inner $r_c$')
ax.plot(ell_range, outer_rc, 's-', ms=4, label='Outer $r_c$')
ax.set_xlabel('Topological charge $\\ell$'); ax.set_ylabel('$r_c / r_{core}$')
ax.set_title('(d) Critical radii vs charge'); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 2. Lyapunov Exponent Across the Vortex Profile

The Lyapunov exponent $\lambda(r)$ of the circle map at local coupling $K(r)$ and
the golden-mean rotation number $\Omega = \phi^{-1}$ shows the radial transition:
$\lambda > 0$ (chaotic) in the overcritical core, $\lambda \leq 0$ (mode-locked)
in the exterior. The red dashed lines mark the critical radii.

In [ ]:
Omega_golden = (np.sqrt(5) - 1) / 2
r_lyap = np.linspace(0.02, 4*r_core, 80)

fig, ax = plt.subplots(figsize=(10, 5))
for ell in [1, 3, 8]:
    lyap = [circle_map_lyapunov(Omega_golden, regime_map(ri, ell, wavelength, **kw)) for ri in r_lyap]
    ax.plot(r_lyap/r_core, lyap, label=f'$\\ell = {ell}$')

ax.axhline(0, color='black', ls='-', lw=0.5)
for rc in find_critical_radii(1, wavelength, **kw):
    ax.axvline(rc/r_core, color='red', ls='--', alpha=0.5)
ax.set_xlabel('$r / r_{core}$'); ax.set_ylabel('$\\lambda(r)$')
ax.set_title('Lyapunov exponent at $\\Omega = \\phi^{-1}$ across the vortex profile')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Critical Radius Table

The inner critical radius $r_{c,\text{inner}}$ shrinks as $1/\ell$: higher charge
vortices concentrate more circulation, pushing the peak velocity higher and
narrowing the overcritical core. The outer critical radius (where $1/r$ decay
brings $v$ back below the Stribeck threshold) exists only for low charges.

In [ ]:
print(f"{'ell':>5}  {'r_c,inner/r_core':>16}  {'r_c,outer/r_core':>16}  {'annulus width':>14}")
print("-" * 60)
for ell in range(1, 21):
    cx = find_critical_radii(ell, wavelength, **kw)
    ri = cx[0]/r_core if len(cx) >= 1 else float('nan')
    ro = cx[1]/r_core if len(cx) >= 2 else float('nan')
    w = ro - ri if not (np.isnan(ri) or np.isnan(ro)) else float('nan')
    print(f"{ell:>5}  {ri:>16.4f}  {ro:>16.4f}  {w:>14.4f}")

## 4. Computable Open Question

For a specific photonic crystal (e.g., Pryamikov's 2D array, arXiv 2601.21704):

1. Extract the Stribeck parameters $(K_{\text{static}}, K_{\text{kinetic}}, v_{\text{thr}})$
   from the electromagnetic coupling structure. $K_{\text{static}}$ is the coupling
   at zero Poynting flux (band-gap regime); $K_{\text{kinetic}}$ is the coupling at
   high flux (transmission regime); $v_{\text{thr}}$ is the flux scale at which
   the transition occurs.

2. Compute the Poynting vector field $v(r)$ around each vortex from the full-wave
   simulation.

3. Evaluate $K(r) = K_{\text{stribeck}}(v(r))$ and find $r_c$ where $K(r_c) = 1$.

4. Compare the predicted reversal boundary $r_c$ to the observed boundary of
   reverse Poynting vector flow in the simulation.

If they match, the Stribeck-modified circle map provides the K-derivation for this
system — the bridge between the grammar and the physics. If they do not match, the
Stribeck parameterization is insufficient and a different coupling model is needed.

The key structural prediction that is independent of Stribeck parameters:
the overcritical region (reversed flow) is always centered on the vortex core
(where v = 0, K is maximal), and its extent shrinks monotonically with topological
charge $\ell$.

## 5. Connection to Existing Framework

The computation above uses:
- The **Stribeck curve** (L1 node `stribeck-curve` in the DAG) for K(v)
- The **fold measure** $\mu = \arccos(1/K)/\pi$ (exact, from the overcritical circle map)
- The **Lyapunov exponent** (L1 node `lyapunov-exponent`) as the diagnostic
- The **Kuramoto-Einstein dictionary** (for the gravity-sector analogue: horizon = r = 0 = N = 0)

The Stribeck parameters for the gravity sector are derived: K = G_gamma (spatial Green's
function). For photonic crystals and optical vortices, extracting the Stribeck parameters
from the electromagnetic response is the open physics problem.